# 01 — Bronze Ingestion: Raw Sensor Telemetry

Ingests streaming JSON batches from the simulator into the **Bronze** Delta table.

| Property | Value |
|----------|-------|
| **Medallion Layer** | Bronze (raw, append-only) |
| **Source** | JSON batch files from streaming simulator |
| **Sink** | `bronze_sensor_readings` Delta table |
| **Idempotency** | MERGE on `(unit_id, cycle)` — safe to re-run |
| **Partitioning** | `_ingestion_date` |
| **Unity Catalog** | `mining_ops.predictive_maintenance.bronze_sensor_readings` |

**Mining context:** This layer captures raw telemetry from haul truck engines and
mill drive motors. Schema enforcement prevents silent data corruption when
sensor configurations change during equipment overhauls.

## 1. Configuration

In [0]:
dbutils.widgets.text("streaming_path", "/Volumes/workspace/default/nasa_streaming/", "Streaming JSON path")
dbutils.widgets.text("bronze_table_path", "/Volumes/workspace/default/bronze/sensor_readings", "Bronze Delta path")
dbutils.widgets.text("checkpoint_path", "/Volumes/workspace/default/checkpoints/bronze_ingestion", "Checkpoint path")
dbutils.widgets.text("pipeline_version", "1.0.0", "Pipeline version")

STREAMING_PATH = dbutils.widgets.get("streaming_path")
BRONZE_PATH = dbutils.widgets.get("bronze_table_path")
CHECKPOINT_PATH = dbutils.widgets.get("checkpoint_path")
PIPELINE_VERSION = dbutils.widgets.get("pipeline_version")

print(f"Streaming source: {STREAMING_PATH}")
print(f"Bronze sink:      {BRONZE_PATH}")
print(f"Checkpoint:       {CHECKPOINT_PATH}")
print(f"Pipeline version: {PIPELINE_VERSION}")

Streaming source: /Volumes/workspace/default/nasa_streaming/
Bronze sink:      /Volumes/workspace/default/bronze/sensor_readings
Checkpoint:       /Volumes/workspace/default/checkpoints/bronze_ingestion
Pipeline version: 1.0.0


## 2. Imports & Schema Definition

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)
from pyspark.sql.functions import (
    col,
    current_timestamp,
    lit,
    to_date,
)
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()

SENSOR_SCHEMA = StructType([
    StructField("unit_id", IntegerType(), nullable=False),
    StructField("cycle", IntegerType(), nullable=False),
    StructField("op_setting_1", DoubleType(), nullable=True),
    StructField("op_setting_2", DoubleType(), nullable=True),
    StructField("op_setting_3", DoubleType(), nullable=True),
    *[StructField(f"s{i}", DoubleType(), nullable=True) for i in range(1, 22)],
    StructField("timestamp", StringType(), nullable=True),
])

print(f"Schema fields: {len(SENSOR_SCHEMA.fields)}")

Schema fields: 27


## 3. Read Streaming JSON with Auto Loader

Auto Loader (`cloudFiles`) provides:
- **Incremental processing**: only new files are ingested each run
- **Schema inference + evolution**: adapts if upstream adds fields
- **Checkpointing**: exactly-once guarantees via checkpoint directory

> **Community Edition fallback:** If `cloudFiles` is unavailable, we fall back
> to a standard `spark.read.json` batch read.

In [0]:
try:
    raw_df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", f"{CHECKPOINT_PATH}/schema")
        .schema(SENSOR_SCHEMA)
        .load(STREAMING_PATH)
    )
    IS_STREAMING = True
    print("Auto Loader stream initialized successfully.")
except Exception as e:
    print(f"Auto Loader not available ({type(e).__name__}), falling back to batch read.")
    raw_df = (
        spark.read
        .schema(SENSOR_SCHEMA)
        .option("multiLine", True)
        .json(STREAMING_PATH)
    )
    IS_STREAMING = False

Auto Loader stream initialized successfully.


## 4. Transform: Add Metadata Columns

In [0]:
if IS_STREAMING:
    bronze_df = raw_df
else:
    bronze_df = raw_df

bronze_df = (
    bronze_df
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_pipeline_version", lit(PIPELINE_VERSION))
    .withColumn("_ingestion_date", to_date(current_timestamp()))
)

if not IS_STREAMING:
    record_count = bronze_df.count()
    print(f"Records read: {record_count}")
    bronze_df.printSchema()

## 5. Write to Bronze Delta

### Streaming path
Uses `foreachBatch` with MERGE for idempotent upserts. Each micro-batch
is merged on the natural key `(unit_id, cycle)` so replayed files do not
create duplicates.

### Batch path (Community Edition)
Falls back to a full MERGE if the table exists, or an initial overwrite.

In [0]:
def merge_to_bronze(batch_df, batch_id):
    """Upsert a micro-batch into the Bronze Delta table.

    Args:
        batch_df: Micro-batch DataFrame from Structured Streaming.
        batch_id: Spark-assigned batch identifier.
    """
    if batch_df.isEmpty():
        print(f"Batch {batch_id}: empty — skipping.")
        return

    if DeltaTable.isDeltaTable(spark, BRONZE_PATH):
        bronze_table = DeltaTable.forPath(spark, BRONZE_PATH)
        (
            bronze_table.alias("target")
            .merge(
                batch_df.alias("source"),
                "target.unit_id = source.unit_id AND target.cycle = source.cycle",
            )
            .whenNotMatchedInsertAll()
            .execute()
        )
        print(f"Batch {batch_id}: MERGE complete — {batch_df.count()} records processed.")
    else:
        (
            batch_df.write
            .format("delta")
            .mode("overwrite")
            .partitionBy("_ingestion_date")
            .option("overwriteSchema", "true")
            .save(BRONZE_PATH)
        )
        print(f"Batch {batch_id}: Initial Bronze table created with {batch_df.count()} records.")


if IS_STREAMING:
    stream_query = (
        bronze_df.writeStream
        .foreachBatch(merge_to_bronze)
        .option("checkpointLocation", CHECKPOINT_PATH)
        .trigger(availableNow=True)
        .start()
    )
    stream_query.awaitTermination()
    print("Streaming ingestion complete.")
else:
    if DeltaTable.isDeltaTable(spark, BRONZE_PATH):
        bronze_table = DeltaTable.forPath(spark, BRONZE_PATH)
        (
            bronze_table.alias("target")
            .merge(
                bronze_df.alias("source"),
                "target.unit_id = source.unit_id AND target.cycle = source.cycle",
            )
            .whenNotMatchedInsertAll()
            .execute()
        )
        print("Batch MERGE complete — new records inserted, duplicates skipped.")
    else:
        (
            bronze_df.write
            .format("delta")
            .mode("overwrite")
            .partitionBy("_ingestion_date")
            .option("overwriteSchema", "true")
            .save(BRONZE_PATH)
        )
        print("Initial Bronze table created.")

Streaming ingestion complete.


## 6. Validate

In [0]:
bronze_result = spark.read.format("delta").load(BRONZE_PATH)

row_count = bronze_result.count()
unit_count = bronze_result.select("unit_id").distinct().count()
partition_count = bronze_result.select("_ingestion_date").distinct().count()

print("=" * 60)
print("BRONZE INGESTION SUMMARY")
print("=" * 60)
print(f"  Total records:        {row_count:,}")
print(f"  Unique units:         {unit_count}")
print(f"  Ingestion partitions: {partition_count}")
print(f"  Pipeline version:     {PIPELINE_VERSION}")
print("=" * 60)

bronze_result.printSchema()

BRONZE INGESTION SUMMARY
  Total records:        900
  Unique units:         5
  Ingestion partitions: 1
  Pipeline version:     1.0.0
root
 |-- unit_id: integer (nullable = true)
 |-- cycle: integer (nullable = true)
 |-- op_setting_1: double (nullable = true)
 |-- op_setting_2: double (nullable = true)
 |-- op_setting_3: double (nullable = true)
 |-- s1: double (nullable = true)
 |-- s2: double (nullable = true)
 |-- s3: double (nullable = true)
 |-- s4: double (nullable = true)
 |-- s5: double (nullable = true)
 |-- s6: double (nullable = true)
 |-- s7: double (nullable = true)
 |-- s8: double (nullable = true)
 |-- s9: double (nullable = true)
 |-- s10: double (nullable = true)
 |-- s11: double (nullable = true)
 |-- s12: double (nullable = true)
 |-- s13: double (nullable = true)
 |-- s14: double (nullable = true)
 |-- s15: double (nullable = true)
 |-- s16: double (nullable = true)
 |-- s17: double (nullable = true)
 |-- s18: double (nullable = true)
 |-- s19: double (nullable = 

## 7. Sample Data

In [0]:
display(bronze_result.orderBy("unit_id", "cycle").limit(20))

unit_id,cycle,op_setting_1,op_setting_2,op_setting_3,s1,s2,s3,s4,s5,s6,s7,s8,s9,s10,s11,s12,s13,s14,s15,s16,s17,s18,s19,s20,s21,timestamp,_ingestion_timestamp,_source_file,_pipeline_version,_ingestion_date
1,1,-7.0E-4,-4.0E-4,100.0,518.67,641.82,1589.7,1400.6,14.62,21.61,554.36,2388.06,9046.19,1.3,47.47,521.66,2388.02,8138.62,8.4195,0.03,392.0,2388.0,100.0,39.06,23.419,2026-04-18T09:35:40.312116+00:00,2026-04-18T10:18:24.020Z,/Volumes/workspace/default/nasa_streaming/batch_000001.json,1.0.0,2026-04-18
1,2,0.0019,-3.0E-4,100.0,518.67,642.15,1591.82,1403.14,14.62,21.61,553.75,2388.04,9044.07,1.3,47.49,522.28,2388.07,8131.49,8.4318,0.03,392.0,2388.0,100.0,39.0,23.4236,2026-04-18T09:35:40.312116+00:00,2026-04-18T10:18:24.020Z,/Volumes/workspace/default/nasa_streaming/batch_000001.json,1.0.0,2026-04-18
1,3,-0.0043,3.0E-4,100.0,518.67,642.35,1587.99,1404.2,14.62,21.61,554.26,2388.08,9052.94,1.3,47.27,522.42,2388.03,8133.23,8.4178,0.03,390.0,2388.0,100.0,38.95,23.3442,2026-04-18T09:35:40.312116+00:00,2026-04-18T10:18:24.020Z,/Volumes/workspace/default/nasa_streaming/batch_000001.json,1.0.0,2026-04-18
1,4,7.0E-4,0.0,100.0,518.67,642.35,1582.79,1401.87,14.62,21.61,554.45,2388.11,9049.48,1.3,47.13,522.86,2388.08,8133.83,8.3682,0.03,392.0,2388.0,100.0,38.88,23.3739,2026-04-18T09:35:40.312116+00:00,2026-04-18T10:18:24.020Z,/Volumes/workspace/default/nasa_streaming/batch_000001.json,1.0.0,2026-04-18
1,5,-0.0019,-2.0E-4,100.0,518.67,642.37,1582.85,1406.22,14.62,21.61,554.0,2388.06,9055.15,1.3,47.28,522.19,2388.04,8133.8,8.4294,0.03,393.0,2388.0,100.0,38.9,23.4044,2026-04-18T09:35:40.312116+00:00,2026-04-18T10:18:24.020Z,/Volumes/workspace/default/nasa_streaming/batch_000001.json,1.0.0,2026-04-18
1,6,-0.0043,-1.0E-4,100.0,518.67,642.1,1584.47,1398.37,14.62,21.61,554.67,2388.02,9049.68,1.3,47.16,521.68,2388.03,8132.85,8.4108,0.03,391.0,2388.0,100.0,38.98,23.3669,2026-04-18T09:35:40.312116+00:00,2026-04-18T10:18:24.020Z,/Volumes/workspace/default/nasa_streaming/batch_000001.json,1.0.0,2026-04-18
1,7,0.001,1.0E-4,100.0,518.67,642.48,1592.32,1397.77,14.62,21.61,554.34,2388.02,9059.13,1.3,47.36,522.32,2388.03,8132.32,8.3974,0.03,392.0,2388.0,100.0,39.1,23.3774,2026-04-18T09:35:40.312116+00:00,2026-04-18T10:18:24.020Z,/Volumes/workspace/default/nasa_streaming/batch_000001.json,1.0.0,2026-04-18
1,8,-0.0034,3.0E-4,100.0,518.67,642.56,1582.96,1400.97,14.62,21.61,553.85,2388.0,9040.8,1.3,47.24,522.47,2388.03,8131.07,8.4076,0.03,391.0,2388.0,100.0,38.97,23.3106,2026-04-18T09:35:40.312116+00:00,2026-04-18T10:18:24.020Z,/Volumes/workspace/default/nasa_streaming/batch_000001.json,1.0.0,2026-04-18
1,9,8.0E-4,1.0E-4,100.0,518.67,642.12,1590.98,1394.8,14.62,21.61,553.69,2388.05,9046.46,1.3,47.29,521.79,2388.05,8125.69,8.3728,0.03,392.0,2388.0,100.0,39.05,23.4066,2026-04-18T09:35:40.312116+00:00,2026-04-18T10:18:24.020Z,/Volumes/workspace/default/nasa_streaming/batch_000001.json,1.0.0,2026-04-18
1,10,-0.0033,1.0E-4,100.0,518.67,641.71,1591.24,1400.46,14.62,21.61,553.59,2388.05,9051.7,1.3,47.03,521.79,2388.06,8129.38,8.4286,0.03,393.0,2388.0,100.0,38.95,23.4694,2026-04-18T09:35:40.312116+00:00,2026-04-18T10:18:24.020Z,/Volumes/workspace/default/nasa_streaming/batch_000001.json,1.0.0,2026-04-18


## 8. Delta Table History

Full audit trail — critical for mining regulatory compliance.

In [0]:
delta_table = DeltaTable.forPath(spark, BRONZE_PATH)
display(delta_table.history(10))

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-04-18T10:18:53.000Z,73183478916021,catherine.varas.padilla@gmail.com,MERGE,"Map(predicate -> [""((unit_id#11652 = unit_id#10982) AND (cycle#11653 = cycle#10983))""], clusterBy -> [], matchedPredicates -> [], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3623011467706618),null,0418-101458-ondslzg3-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 48781, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 3838, materializeSourceTimeMs -> 25, numTargetRowsInserted -> 800, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 0, numTargetRowsUpdated -> 0, numOutputRows -> 800, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 800, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 3643)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
0,2026-04-18T10:18:44.000Z,73183478916021,catherine.varas.padilla@gmail.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [""_ingestion_date""])",null,List(3623011467706618),null,0418-101458-ondslzg3-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 100, numOutputBytes -> 14911)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
